In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

House_Prices = pd.read_csv("/content/House_Prices.csv")
# check dublicate without address
duplicates_colums=  [
    col for col in House_Prices.columns if col != "ADDRESS"

    ]
print("the dublicate ignore the  ADDRESS:"
      ,House_Prices.duplicated(subset=duplicates_colums).sum()
      )
# Remove duplicates
House_Prices=House_Prices.drop_duplicates(subset=duplicates_colums).copy()

# Convert DATE_SOLD from month-year format
House_Prices["DATE_SOLD"] = pd.to_datetime(
    House_Prices["DATE_SOLD"].str.strip(),
    format="%m-%Y",
    errors="coerce"
)

# Extract year and month
House_Prices["sale year"] = House_Prices["DATE_SOLD"].dt.year
House_Prices["sale month"] = House_Prices["DATE_SOLD"].dt.month

# Drop the original date column
House_Prices = House_Prices.drop(columns=["DATE_SOLD"])
# Remove ADDRESS
House_Prices=House_Prices.drop(columns=["ADDRESS"])
# row without the target connt be ,used super learing value
House_Prices=House_Prices.dropna(
    subset=["PRICE"]
    ).copy()
# Separate features and target.
x=House_Prices.drop(columns=["PRICE"])
y=House_Prices["PRICE"]
#split the data

x_train,x_test,y_train,y_test= train_test_split(
 x,y,test_size=0.2,random_state=42
    )
# Identify categorical columns for frequency encoding,Identify numeric columns.
categorical_columns = [
    "SUBURB",
    "NEAREST_STN",
    "NEAREST_SCH"
]

numeric_columns=x_train.select_dtypes(
    include="number"
    ).columns.tolist()
#imputation values,medain
median_columns= [col for col in ["GARAGE", "BUILD_YEAR", "NEAREST_SCH_RANK"]
if col in numeric_columns
]

mean_columns=[col for col in numeric_columns if col not in median_columns]
imputation_values = {}

for col in median_columns:
    imputation_values[col] = x_train[col].median()

for col in mean_columns:
    imputation_values[col] = x_train[col].mean()


# Apply training-derived values to both sets.
x_train=x_train.fillna(imputation_values)
x_test=x_test.fillna(imputation_values)

#process the missing value in the numeric columns

frequencies_mapping={}

for col in categorical_columns:
    if col not in x_train.columns:
        continue

    # Fill missing categories using the training mode
    mode_values = x_train[col].mode(dropna=True)

    if not mode_values.empty:
        fill_values = mode_values.iloc[0]
        x_train[col] = x_train[col].fillna(fill_values)
        x_test[col] = x_test[col].fillna(fill_values)

    # Frequency encoding
    frequencies = x_train[col].value_counts(normalize=True)
    frequencies_mapping = frequencies.to_dict()

    x_train[col] = x_train[col].map(frequencies_mapping).fillna(0)
    x_test[col] = x_test[col].map(frequencies_mapping).fillna(0)


print("Training missing values:")
print(x_train.isnull().sum().sum())

print("Testing missing values:")
print(x_test.isnull().sum().sum())

print("Training shape:", x_train.shape)
print("Testing shape:", x_test.shape)

print("\nFirst five training rows:")
print(x_train.head())

#ML_ASSIGMENT 2

print(x_train.isnull().sum())
print(x_test.isnull().sum())
# Convert categorical columns to numeric
x_train = pd.get_dummies(x_train, drop_first=True)
x_test = pd.get_dummies(x_test, drop_first=True)

# Make sure both datasets have the same columns
x_train, x_test = x_train.align(
    x_test, join="left", axis=1, fill_value=0
)

# Replace infinite values with NaN
x_train = x_train.replace([np.inf, -np.inf], np.nan)
x_test = x_test.replace([np.inf, -np.inf], np.nan)

# Remove columns that are completely empty in training data
empty_columns = x_train.columns[x_train.isnull().all()]
x_train = x_train.drop(columns=empty_columns)
x_test = x_test.drop(columns=empty_columns, errors="ignore")

# Fill missing values using training medians
medians = x_train.median()
x_train = x_train.fillna(medians)
x_test = x_test.fillna(medians)

# Train the model
model = LinearRegression()
model.fit(x_train, y_train)

# Predict prices
y_pred = model.predict(x_test)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Linear Regression RMSE:", rmse)

the dublicate ignore the  ADDRESS: 0
Training missing values:
0
Testing missing values:
0
Training shape: (26924, 18)
Testing shape: (6732, 18)

First five training rows:
         SUBURB  BEDROOMS  BATHROOMS  GARAGE  LAND_AREA  FLOOR_AREA  \
9451   0.004308         4          2     2.0        866         200   
6534   0.005460         4          2     2.0        447         199   
431    0.001931         5          3     2.0      60024         236   
11544  0.003417         4          2     2.0        260         157   
12785  0.004308         4          2     2.0        423         285   

       BUILD_YEAR  CBD_DIST  NEAREST_STN  NEAREST_STN_DIST  POSTCODE  \
9451       1991.0     21400     0.019202              4000      6025   
6534       2011.0     38700     0.034245               379      6036   
431        1990.0     24000     0.024848             12800      6076   
11544      2000.0     16000     0.023362              2600      6162   
12785      2011.0      8000     0.025962  